<a href="https://colab.research.google.com/github/sofialindner/ntl-urban-expansion-mapping/blob/develop/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import geopandas as gpd

# Lê shapefile
cities = gpd.read_file("LITORAL_SC_REPROJETADO.shp")

# Normaliza tipo de cada atributo
cities["CD_MUN"] = cities["CD_MUN"].astype(int)
cities["NM_MUN"] = cities["NM_MUN"].astype(str)

# Indexa dados de cada município por código
cities_by_code = {
    row["CD_MUN"]: {
        "name": row["NM_MUN"],
        "geometry": row.geometry,
        "row": row
    }
    for _, row in cities.iterrows()
}

In [2]:
# Intervalo de anos
YEARS = ["2015", "2016", "2017", "2018", "2019", "2020", "2021", "2022", "2023", "2024", "2025"]

# Intervalo de meses
MONTHS = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"]

# Lista de cidades separados por região
CITIES_BY_REGION = {
  "blumenau": [
    4202008,
    4202107,
    4202453,
    4208203,
    4208302,
    4211306,
    4212502,
    4212809,
    4213500,
    4218004,
  ],
  "florianopolis": [
    4202305,
    4205407,
    4205704,
    4206009,
    4207304,
    4211900,
    4212304,
    4216602,
  ],
  "joinville": [4201307, 4202057, 4208450, 4216206],
  "criciuma": [4220000, 4201406, 4201950, 4202073, 4212254, 4208807, 4209409],
}

In [ ]:
from huggingface_hub import snapshot_download
from glob import glob
import rasterio

# Baixa dataset inteiro mantendo estrutura
dataset_path = snapshot_download(
    repo_id="incrisvel/ntl-urban-expansion",
    repo_type="dataset"
)

In [ ]:
import pickle
import rasterio
from rasterio.mask import mask
import numpy as np
from glob import glob
from pathlib import Path
import re

base_path = Path(dataset_path)

# Diretório para persistência
output_dir = Path("processed/series")
output_dir.mkdir(exist_ok=True)

# Utlitário para indexação
def timestamp_from_path(path):
    filename = Path(path).stem

    match = re.search(r'_(\d{4})_(\d{2})_', filename)
    if not match:
        raise ValueError(
            f"Não foi possível extrair ano/mês de {filename}"
        )

    year = match.group(1)
    month = match.group(2)

    return f"{year}-{month}"


for region_name, city_codes in CITIES_BY_REGION.items():

    print(f"\nREGIÃO: {region_name}")

    # Estrutura da região inteira
    pixel_series_by_city = {
        city_code: {}
        for city_code in city_codes
    }

    for year in YEARS:

        image_paths = sorted(
            glob(str(base_path / f"{region_name}/{year}/*.tif"))
        )

        for path in image_paths:
            print(path)

            # Extrai timestamp
            timestamp = timestamp_from_path(path)

            # Abre raster da região
            with rasterio.open(path) as src:

                for city_code in city_codes:

                    city = cities_by_code[city_code]

                    # Extrai atributos do município
                    city_name = city["name"]
                    city_geometry = city["geometry"]

                    print(f"  Município: {city_name}")

                    try:
                       # Recorta raster pelo município
                        out_image, out_transform = mask(
                            src,
                            [city_geometry],
                            crop=True
                        )

                    except ValueError as e:

                        # Município completamente fora da imagem
                        if "Input shapes do not overlap raster" in str(e):
                            continue

                    img = out_image[0]

                    # Verifica valor usado no .tif para nodata
                    nodata = src.nodata

                    valid_mask = np.isfinite(img)

                    # Filtra valores nodata
                    if nodata is not None:
                        valid_mask &= (img != nodata)

                    rows, cols = np.where(valid_mask)

                    for row, col in zip(rows, cols):

                        value = img[row, col]

                        if not np.isfinite(value):
                            continue

                        # Coordenadas geográficas reais
                        x, y = rasterio.transform.xy(
                            out_transform,
                            row,
                            col
                        )

                        # Arredonda coordenadas para evitar diferenças de ponto flutuante
                        key = (
                            round(x, 6),
                            round(y, 6)
                        )

                        # Cria chave do pixel a partir das coordenadas
                        if key not in pixel_series_by_city[city_code]:
                            pixel_series_by_city[city_code][key] = {}

                        # Armazena luminosidade indexada por ano-mês
                        pixel_series_by_city[city_code][key][timestamp] = float(value)

    # Persistência ao final da região
    for city_code in city_codes:

        city = cities_by_code[city_code]

        city_name = city["name"]

        output_path = output_dir / f"{city_code}_{city_name}.pkl"

        with open(output_path, "wb") as f:
            pickle.dump(
                {
                    "city_code": city_code,
                    "city_name": city_name,
                    "pixels": pixel_series_by_city[city_code]
                },
                f,
                protocol=pickle.HIGHEST_PROTOCOL
            )

        print(f"Salvo: {output_path}")

In [13]:
import numpy as np
from sklearn.linear_model import LinearRegression


def pixel_linear_trend(serie):
    """
    Calcula a inclinação de uma série temporal e permite identificar:
    - Crescimento
    - Estabilidade
    - Decaimento
    """
    y = np.array(serie, dtype=float)

    # Filtra valores NaN
    valid = ~np.isnan(y)
    y = y[valid]

    # Não tenta calcular regressão para séries pequenas
    if len(y) < 2:
        return 0.0

    X = np.arange(len(y)).reshape(-1, 1)

    model = LinearRegression()
    model.fit(X, y)

    return float(model.coef_[0])


def pixel_seasonality(pixel_data):
    """
    Mede sazonalidade através da diferença entre
    luminosidade média do verão e do inverno.
    """
    SUMMER_MONTHS = [12, 1, 2]
    WINTER_MONTHS = [6, 7, 8]

    summer = []
    winter = []

    for timestamp, value in pixel_data.items():

        month = int(timestamp.split("-")[1])

        if month in SUMMER_MONTHS:
            summer.append(value)

        elif month in WINTER_MONTHS:
            winter.append(value)

    if len(summer) == 0 or len(winter) == 0:
        return 0.0

    return float(np.mean(summer) - np.mean(winter))


def pixel_growth_rate(serie):
    """
    Mede crescimento acumulado na série temporal.
    """
    if len(serie) < 2:
        return 0.0

    first = serie[0]
    last = serie[-1]

    if first == 0:
        return 0.0

    return (last - first) / first

In [14]:
from pathlib import Path
import pickle
import numpy as np

series_dir = Path("processed/series")
features_dir = Path("processed/features")

features_dir.mkdir(
    parents=True,
    exist_ok=True
)

for city_pickle in series_dir.glob("*.pkl"):

    print(city_pickle.name)

    with open(city_pickle, "rb") as f:
        city_data = pickle.load(f)

    pixel_features = {}

    for position, pixel_data in city_data["pixels"].items():

        ordered_values = [
            pixel_data[timestamp]
            for timestamp in sorted(pixel_data.keys())
        ]

        serie = np.array(
            ordered_values,
            dtype=float
        )

        if len(serie) < 2:
            continue

        pixel_features[position] = {
            "mean": np.mean(serie),
            "var": np.var(serie),
            "max": np.max(serie),
            "min": np.min(serie),
            "linear_trend": pixel_linear_trend(serie),
            "seasonality": pixel_seasonality(pixel_data),
            "growth_rate": pixel_growth_rate(serie),
            "position": position
        }

    output = {
        "city_code": city_data["city_code"],
        "city_name": city_data["city_name"],
        "features": pixel_features
    }

    output_path = (
        features_dir /
        city_pickle.name
    )

    with open(output_path, "wb") as f:
        pickle.dump(
            output,
            f,
            protocol=pickle.HIGHEST_PROTOCOL
        )

4216206_SÃ£o Francisco do Sul.pkl
4205407_FlorianÃ³polis.pkl
4213500_Porto Belo.pkl
4201950_BalneÃ¡rio Arroio do Silva.pkl
4209409_Laguna.pkl
4201307_Araquari.pkl
4202008_BalneÃ¡rio CamboriÃº.pkl
4202107_Barra Velha.pkl
4220000_BalneÃ¡rio RincÃ£o.pkl
4201406_AraranguÃ¡.pkl
4216602_SÃ£o JosÃ©.pkl
4202057_BalneÃ¡rio Barra do Sul.pkl
4205704_Garopaba.pkl
4212502_Penha.pkl
4208203_ItajaÃ­.pkl
4218004_Tijucas.pkl
4208807_Jaguaruna.pkl
4211306_Navegantes.pkl
4202453_Bombinhas.pkl
4202305_BiguaÃ§u.pkl
4211900_PalhoÃ§a.pkl
4208450_ItapoÃ¡.pkl
4207304_Imbituba.pkl
4212304_Paulo Lopes.pkl
4212809_BalneÃ¡rio PiÃ§arras.pkl
4208302_Itapema.pkl
4212254_Passo de Torres.pkl
4202073_BalneÃ¡rio Gaivota.pkl
4206009_Governador Celso Ramos.pkl


In [29]:
from pathlib import Path

summary_dir = Path("processed/summaries")
summary_dir.mkdir(
    parents=True,
    exist_ok=True
)

def export_cluster_summary(cluster_summary):
  summary_export = cluster_summary.copy()

  summary_export["category"] = (
      summary_export.index
      .map(cluster_labels)
  )

  summary_export.to_csv(
      summary_dir /
      f"{city_data['city_code']}_{city_data['city_name']}.csv"
  )

In [33]:
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import pandas as pd

clusters_dir = Path("processed/clusters")

clusters_dir.mkdir(
    parents=True,
    exist_ok=True
)

for feature_pickle in features_dir.glob("*.pkl"):

    print(feature_pickle.name)

    with open(feature_pickle, "rb") as f:
        city_data = pickle.load(f)

    df = pd.DataFrame(city_data["features"].values())

    if len(df) < 50:
        continue

    # Remove pixels sem atividade e ruídos
    df = df[df["mean"] > 1]
    df = df[df["max"] > 2]

    X = df[
        [
            "mean",
            "var",
            "linear_trend",
            "seasonality",
            "growth_rate"
        ]
    ]

    # Normaliza os valores de cada feature
    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(X)

    # Calcula KMeans para o município
    kmeans = KMeans(
        n_clusters=4,
        random_state=1,
        n_init="auto"
    )

    df["cluster"] = kmeans.fit_predict(X_scaled)

    cluster_summary = (
        df.groupby("cluster")
          .agg({
              "mean": "mean",
              "var": "mean",
              "linear_trend": "mean",
              "growth_rate": "mean",
              "seasonality": "mean"
          })
          .round(3)
    )

    cluster_summary["expansion_score"] = (
        cluster_summary["linear_trend"] +
        cluster_summary["growth_rate"]
    )

    cluster_summary["pixel_count"] = (
        df.groupby("cluster")
          .size()
    )

    # Conjunto para controlar quais clusters já receberam label
    assigned = set()

    # Encontra cluster em EXPANSÃO
    expansion_cluster = (
        cluster_summary["expansion_score"]
        .idxmax()
    )
    assigned.add(expansion_cluster)

    # Encontra cluster SAZONAL
    seasonal_candidates = (
        cluster_summary
        .drop(index=list(assigned))
    )
    seasonal_cluster = (
        seasonal_candidates["seasonality"]
        .idxmax()
    )
    assigned.add(seasonal_cluster)

    # Encontra cluster CONSOLIDADO
    urban_candidates = (
        cluster_summary
        .drop(index=list(assigned))
    )
    urban_cluster = (
        urban_candidates["mean"]
        .idxmax()
    )
    assigned.add(urban_cluster)

    # Encontra cluster com BAIXA ATIVIDADE
    inactive_cluster = (
        cluster_summary
        .drop(index=list(assigned))
        .index[0]
    )

    # Atribui labels ao índice de cada cluster
    cluster_labels = {
        inactive_cluster: "LOW_ACTIVITY",
        urban_cluster: "URBAN",
        expansion_cluster: "EXPANSION",
        seasonal_cluster: "SEASONAL"
    }
    df["cluster_type"] = df["cluster"].map(cluster_labels)

    cluster_result = {
        "city_code": city_data["city_code"],
        "city_name": city_data["city_name"],
        "cluster_summary": cluster_summary,
        "cluster_labels": cluster_labels,

        "expansion_cluster": expansion_cluster,
        "seasonal_cluster": seasonal_cluster,
        "urban_cluster": urban_cluster,
        "inactive_cluster": inactive_cluster,

        "data": df
    }

    # Salva interpretação dos clusters
    export_cluster_summary(cluster_summary)

    output_path = (clusters_dir / feature_pickle.name)

    # Salva pickle do cluster
    with open(output_path, "wb") as f:
        pickle.dump(
            cluster_result,
            f,
            protocol=pickle.HIGHEST_PROTOCOL
        )

4216206_SÃ£o Francisco do Sul.pkl
4205407_FlorianÃ³polis.pkl
4213500_Porto Belo.pkl
4201950_BalneÃ¡rio Arroio do Silva.pkl
4209409_Laguna.pkl
4201307_Araquari.pkl
4202008_BalneÃ¡rio CamboriÃº.pkl
4202107_Barra Velha.pkl
4220000_BalneÃ¡rio RincÃ£o.pkl
4201406_AraranguÃ¡.pkl
4216602_SÃ£o JosÃ©.pkl
4202057_BalneÃ¡rio Barra do Sul.pkl
4205704_Garopaba.pkl
4212502_Penha.pkl
4208203_ItajaÃ­.pkl
4218004_Tijucas.pkl
4208807_Jaguaruna.pkl
4211306_Navegantes.pkl
4202453_Bombinhas.pkl
4202305_BiguaÃ§u.pkl
4211900_PalhoÃ§a.pkl
4208450_ItapoÃ¡.pkl
4207304_Imbituba.pkl
4212304_Paulo Lopes.pkl
4212809_BalneÃ¡rio PiÃ§arras.pkl
4208302_Itapema.pkl
4212254_Passo de Torres.pkl
4202073_BalneÃ¡rio Gaivota.pkl
4206009_Governador Celso Ramos.pkl


In [34]:
from pathlib import Path
import pickle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

clusters_dir = Path("processed/clusters")
plots_dir = Path("processed/plots")

plots_dir.mkdir(
    parents=True,
    exist_ok=True
)

CATEGORY_COLORS = {
    "LOW_ACTIVITY": 0,
    "URBAN": 1,
    "EXPANSION": 2,
    "SEASONAL": 3
}

cluster_cmap = ListedColormap([
    "#d3d3d3",  # LOW_ACTIVITY
    "#1f77b4",  # URBAN
    "#2ca02c",  # EXPANSION
    "#ff7f0e",  # SEASONAL
])

for cluster_file in sorted(clusters_dir.glob("*.pkl")):

    print(f"Gerando mapa: {cluster_file.name}")

    with open(cluster_file, "rb") as f:
        city_data = pickle.load(f)

    df = city_data["data"].copy()

    # Recupera coordenadas
    df["x"] = df["position"].apply(lambda p: p[0])
    df["y"] = df["position"].apply(lambda p: p[1])

    # Reconstrói grade espacial
    x_unique = np.sort(df["x"].unique())
    y_unique = np.sort(df["y"].unique())

    cluster_img = np.full(
        (len(y_unique), len(x_unique)),
        np.nan
    )

    x_index = {
        value: idx
        for idx, value in enumerate(x_unique)
    }

    y_index = {
        value: idx
        for idx, value in enumerate(y_unique)
    }

    for _, row in df.iterrows():

        xi = x_index[row["x"]]
        yi = y_index[row["y"]]

        cluster_img[yi, xi] = CATEGORY_COLORS[row["cluster_type"]]

    # Cria figura
    fig, ax = plt.subplots(
        figsize=(10, 10)
    )

    im = ax.imshow(
        cluster_img,
        cmap=cluster_cmap,
        interpolation="nearest",
        origin="upper",
        vmin=0,
        vmax=3
    )

    ax.set_title(
        f"{city_data['city_name']} - KMeans (4 clusters)",
        fontsize=14,
        pad=12
    )

    ax.axis("off")

    cbar = plt.colorbar(
        im,
        ax=ax,
        shrink=0.8
    )

    cbar.set_ticks([0, 1, 2, 3])
    cbar.set_ticklabels([
        "LOW_ACTIVITY",
        "URBAN",
        "EXPANSION",
        "SEASONAL"
    ])

    output_path = (
        plots_dir /
        f"{city_data['city_code']}_{city_data['city_name']}.png"
    )

    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print(f"Salvo em: {output_path}")

Gerando mapa: 4201307_Araquari.pkl
Salvo em: processed/plots/4201307_Araquari.png
Gerando mapa: 4201406_AraranguÃ¡.pkl
Salvo em: processed/plots/4201406_AraranguÃ¡.png
Gerando mapa: 4201950_BalneÃ¡rio Arroio do Silva.pkl
Salvo em: processed/plots/4201950_BalneÃ¡rio Arroio do Silva.png
Gerando mapa: 4202008_BalneÃ¡rio CamboriÃº.pkl
Salvo em: processed/plots/4202008_BalneÃ¡rio CamboriÃº.png
Gerando mapa: 4202057_BalneÃ¡rio Barra do Sul.pkl
Salvo em: processed/plots/4202057_BalneÃ¡rio Barra do Sul.png
Gerando mapa: 4202073_BalneÃ¡rio Gaivota.pkl
Salvo em: processed/plots/4202073_BalneÃ¡rio Gaivota.png
Gerando mapa: 4202107_Barra Velha.pkl
Salvo em: processed/plots/4202107_Barra Velha.png
Gerando mapa: 4202305_BiguaÃ§u.pkl
Salvo em: processed/plots/4202305_BiguaÃ§u.png
Gerando mapa: 4202453_Bombinhas.pkl
Salvo em: processed/plots/4202453_Bombinhas.png
Gerando mapa: 4205407_FlorianÃ³polis.pkl
Salvo em: processed/plots/4205407_FlorianÃ³polis.png
Gerando mapa: 4205704_Garopaba.pkl
Salvo em: p

In [35]:
# Comprime a pasta de processados para um zip
!zip -r processed.zip /content/processed

# Baixa o zip no computador
from google.colab import files
files.download('processed.zip')

updating: content/processed/ (stored 0%)
updating: content/processed/series/ (stored 0%)
updating: content/processed/series/4216206_SÃ£o Francisco do Sul.pkl (deflated 44%)
updating: content/processed/series/4205407_FlorianÃ³polis.pkl (deflated 45%)
updating: content/processed/series/4213500_Porto Belo.pkl (deflated 44%)
updating: content/processed/series/4201950_BalneÃ¡rio Arroio do Silva.pkl (deflated 48%)
updating: content/processed/series/4209409_Laguna.pkl (deflated 45%)
updating: content/processed/series/4201307_Araquari.pkl (deflated 44%)
updating: content/processed/series/4202008_BalneÃ¡rio CamboriÃº.pkl (deflated 45%)
updating: content/processed/series/4202107_Barra Velha.pkl (deflated 45%)
updating: content/processed/series/4220000_BalneÃ¡rio RincÃ£o.pkl (deflated 46%)
updating: content/processed/series/4201406_AraranguÃ¡.pkl (deflated 44%)
updating: content/processed/series/4216602_SÃ£o JosÃ©.pkl (deflated 45%)
updating: content/processed/series/4202057_BalneÃ¡rio Barra do S

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>